<a href="https://colab.research.google.com/github/Bhanu-Jakka/ML_Project/blob/main/Feature_Engineering_IPL_MatchPrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
## Imports
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [2]:
## Load data
matches = pd.read_csv('matches.csv')
deliveries = pd.read_csv('deliveries.csv')

print("Matches:", matches.shape)
print("Deliveries:", deliveries.shape)

Matches: (1095, 20)
Deliveries: (260920, 17)


In [3]:
## Standardize team names
rename_map = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Deccan Chargers': 'Sunrisers Hyderabad',
    'Kings XI Punjab': 'Punjab Kings'
}

for col in ['team1', 'team2', 'toss_winner', 'winner']:
    matches[col] = matches[col].replace(rename_map)

for col in ['batting_team', 'bowling_team']:
    deliveries[col] = deliveries[col].replace(rename_map)

In [4]:
## Sort by date, drop ties/no-results
matches['date'] = pd.to_datetime(matches['date'])
matches = matches.sort_values('date').reset_index(drop=True)

matches_clean = matches[matches['winner'].notna()].copy()
matches_clean = matches_clean[matches_clean['result'] != 'no result'] if 'result' in matches_clean.columns else matches_clean

print("Matches before:", matches.shape[0], "-> after dropping ties/no-result:", matches_clean.shape[0])


Matches before: 1095 -> after dropping ties/no-result: 1090


In [5]:
## Per-match team batting/bowling stats from deliveries.csv
match_batting = deliveries.groupby(['match_id','batting_team'])['total_runs'].sum().reset_index()
match_batting.columns = ['id','team','runs_scored']

match_bowling = deliveries.groupby(['match_id','bowling_team'])['total_runs'].sum().reset_index()
match_bowling.columns = ['id','team','runs_conceded']

match_wickets = deliveries.groupby(['match_id','bowling_team'])['is_wicket'].sum().reset_index()
match_wickets.columns = ['id','team','wickets_taken']

team_match_stats = match_batting.merge(match_bowling, on=['id','team'], how='outer') \
                                 .merge(match_wickets, on=['id','team'], how='outer')
team_match_stats = team_match_stats.merge(matches[['id','date']], on='id', how='left')
team_match_stats['date'] = pd.to_datetime(team_match_stats['date'])
team_match_stats = team_match_stats.sort_values('date')

In [6]:
## Helper functions
def get_avg_stat(team, date, stat_col, stats_df, window=5):
    past = stats_df[(stats_df['team'] == team) & (stats_df['date'] < date)].tail(window)
    if len(past) == 0:
        return stats_df[stat_col].mean()
    return past[stat_col].mean()

def get_weighted_form(team, date, matches_df, window=5):
    past = matches_df[((matches_df['team1'] == team) | (matches_df['team2'] == team)) &
                       (matches_df['date'] < date)].tail(window)
    if len(past) == 0:
        return 0.5
    weights = np.linspace(0.5, 1.5, len(past))
    wins = (past['winner'] == team).astype(int).values
    return np.average(wins, weights=weights)

def get_win_streak(team, date, matches_df):
    past = matches_df[((matches_df['team1'] == team) | (matches_df['team2'] == team)) &
                       (matches_df['date'] < date)].sort_values('date', ascending=False)
    streak = 0
    for _, row in past.iterrows():
        if row['winner'] == team:
            streak += 1
        else:
            break
    return streak

def get_h2h_ratio(team1, team2, date, matches_df):
    past = matches_df[(((matches_df['team1'] == team1) & (matches_df['team2'] == team2)) |
                        ((matches_df['team1'] == team2) & (matches_df['team2'] == team1))) &
                       (matches_df['date'] < date)]
    if len(past) == 0:
        return 0.5
    team1_wins = (past['winner'] == team1).sum()
    return team1_wins / len(past)

def get_venue_win_rate(team, venue, date, matches_df):
    past = matches_df[((matches_df['team1'] == team) | (matches_df['team2'] == team)) &
                       (matches_df['venue'] == venue) &
                       (matches_df['date'] < date)]
    if len(past) == 0:
        return 0.5
    wins = (past['winner'] == team).sum()
    return wins / len(past)

def get_days_since_last_match(team, date, matches_df):
    past = matches_df[((matches_df['team1'] == team) | (matches_df['team2'] == team)) &
                       (matches_df['date'] < date)]
    if len(past) == 0:
        return 14
    return (date - past['date'].max()).days

In [7]:
## Team strength features
for prefix, team_col in [('team1', 'team1'), ('team2', 'team2')]:
    matches_clean[f'{prefix}_avg_runs_scored'] = matches_clean.apply(
        lambda r: get_avg_stat(r[team_col], r['date'], 'runs_scored', team_match_stats), axis=1)
    matches_clean[f'{prefix}_avg_runs_conceded'] = matches_clean.apply(
        lambda r: get_avg_stat(r[team_col], r['date'], 'runs_conceded', team_match_stats), axis=1)
    matches_clean[f'{prefix}_avg_wickets_taken'] = matches_clean.apply(
        lambda r: get_avg_stat(r[team_col], r['date'], 'wickets_taken', team_match_stats), axis=1)

In [8]:
##  Weighted recent form
matches_clean['team1_form'] = matches_clean.apply(
    lambda r: get_weighted_form(r['team1'], r['date'], matches_clean), axis=1)
matches_clean['team2_form'] = matches_clean.apply(
    lambda r: get_weighted_form(r['team2'], r['date'], matches_clean), axis=1)

In [9]:
## Win streaks
matches_clean['team1_win_streak'] = matches_clean.apply(
    lambda r: get_win_streak(r['team1'], r['date'], matches_clean), axis=1)
matches_clean['team2_win_streak'] = matches_clean.apply(
    lambda r: get_win_streak(r['team2'], r['date'], matches_clean), axis=1)

In [10]:
## Head-to-head win ratio
matches_clean['h2h_team1_win_ratio'] = matches_clean.apply(
    lambda r: get_h2h_ratio(r['team1'], r['team2'], r['date'], matches_clean), axis=1)

In [11]:
## Home/away advantage
city_home = matches.melt(id_vars=['city'], value_vars=['team1','team2'], value_name='team') \
                    .groupby('team')['city'].agg(lambda x: x.value_counts().idxmax())

matches_clean['team1_home'] = (matches_clean['team1'].map(city_home) == matches_clean['city']).astype(int)
matches_clean['team2_home'] = (matches_clean['team2'].map(city_home) == matches_clean['city']).astype(int)

In [12]:
## Toss features
matches_clean['toss_won_by_team1'] = (matches_clean['toss_winner'] == matches_clean['team1']).astype(int)
matches_clean['toss_decision_bat'] = (matches_clean['toss_decision'] == 'bat').astype(int)

In [13]:
## Venue win rate
matches_clean['team1_venue_form'] = matches_clean.apply(
    lambda r: get_venue_win_rate(r['team1'], r['venue'], r['date'], matches_clean), axis=1)
matches_clean['team2_venue_form'] = matches_clean.apply(
    lambda r: get_venue_win_rate(r['team2'], r['venue'], r['date'], matches_clean), axis=1)

In [14]:
## Rest days
matches_clean['team1_rest_days'] = matches_clean.apply(
    lambda r: get_days_since_last_match(r['team1'], r['date'], matches_clean), axis=1)
matches_clean['team2_rest_days'] = matches_clean.apply(
    lambda r: get_days_since_last_match(r['team2'], r['date'], matches_clean), axis=1)

In [15]:
## Target variable
matches_clean['team1_win'] = (matches_clean['winner'] == matches_clean['team1']).astype(int)

In [16]:
## Final feature set
feature_cols = [
    'team1_form', 'team2_form',
    'team1_win_streak', 'team2_win_streak',
    'h2h_team1_win_ratio',
    'team1_home', 'team2_home',
    'toss_won_by_team1', 'toss_decision_bat',
    'team1_venue_form', 'team2_venue_form',
    'team1_avg_runs_scored', 'team2_avg_runs_scored',
    'team1_avg_runs_conceded', 'team2_avg_runs_conceded',
    'team1_avg_wickets_taken', 'team2_avg_wickets_taken',
    'team1_rest_days', 'team2_rest_days'
]

model_df = matches_clean[feature_cols + ['team1_win']].copy()
model_df = model_df.dropna()
print(model_df.shape)
model_df.head(10)


(1090, 20)


,team1_form,team2_form,team1_win_streak,team2_win_streak,h2h_team1_win_ratio,team1_home,team2_home,toss_won_by_team1,toss_decision_bat,team1_venue_form,team2_venue_form,team1_avg_runs_scored,team2_avg_runs_scored,team1_avg_runs_conceded,team2_avg_runs_conceded,team1_avg_wickets_taken,team2_avg_wickets_taken,team1_rest_days,team2_rest_days,team1_win
0,0.5,0.50,0,0,0.5,1,0,1,0,0.5,0.5,159.010517,159.010517,159.010517,159.010517,5.921353,5.921353,14,14,0
1,0.5,0.50,0,0,0.5,1,0,0,1,0.5,0.5,159.010517,159.010517,159.010517,159.010517,5.921353,5.921353,14,14,0
2,0.5,0.50,0,0,0.5,1,0,0,1,0.5,0.5,159.010517,159.010517,159.010517,159.010517,5.921353,5.921353,14,14,1
3,0.5,0.00,0,0,0.5,1,0,1,1,0.5,0.5,159.010517,82.000000,159.010517,222.000000,5.921353,3.000000,14,2,0
4,1.0,0.50,1,0,0.5,1,0,0,1,0.5,0.5,222.000000,159.010517,82.000000,159.010517,10.000000,5.921353,2,14,1
5,0.0,0.00,0,0,0.5,1,0,0,1,0.5,0.5,129.000000,207.000000,132.000000,240.000000,1.000000,5.000000,2,2,1
6,0.0,1.00,0,1,0.5,1,0,1,1,0.5,0.5,110.000000,132.000000,112.000000,129.000000,5.000000,8.000000,2,3,0
7,1.0,0.00,1,0,0.5,1,0,0,0,0.5,0.5,240.000000,165.000000,207.000000,166.000000,4.000000,5.000000,4,3,1
8,0.0,0.75,0,1,0.5,1,0,0,0,0.0,0.5,126.000000,148.500000,127.500000,149.000000,3.000000,4.500000,2,3,0
9,0.0,0.00,0,0,0.5,1,0,0,0,0.0,0.5,186.500000,183.500000,204.000000,187.000000,4.500000,5.000000,4,2,1


In [17]:
## Correlation with target
correlations = model_df.corr()['team1_win'].sort_values(ascending=False)
print(correlations)

team1_win                  1.000000
team1_home                 0.072990
team2_win_streak           0.061286
toss_decision_bat          0.059042
h2h_team1_win_ratio        0.037752
team1_form                 0.037483
team2_form                 0.036730
team1_avg_wickets_taken    0.027322
team1_avg_runs_scored      0.022804
toss_won_by_team1          0.021162
team2_venue_form           0.008216
team2_avg_wickets_taken    0.007056
team1_venue_form           0.000665
team2_avg_runs_scored      0.000594
team1_win_streak          -0.005936
team2_rest_days           -0.009693
team2_avg_runs_conceded   -0.015804
team1_rest_days           -0.016203
team1_avg_runs_conceded   -0.045165
team2_home                -0.071795
Name: team1_win, dtype: float64


In [18]:
## Save
model_df.to_csv('model_ready_data.csv', index=False)
print("Saved model_ready_data.csv with", len(feature_cols), "features")

Saved model_ready_data.csv with 19 features
